**Preparando a estrutura de Catálogo, Volume (camada Landing) e Schema (camada Bronze)**

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS ecommerce.default.landing

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ecommerce.bronze

**Importação de bibliotecas para manipulação de caminhos, DataFrames do Spark e requisições de API**

In [0]:
import os
from pyspark.sql import functions as F 
import requests

In [0]:
path = "/Volumes/ecommerce/default/landing/"
files = os.listdir(path)

# Dicionário: Nome do arquivo original -> Nome da tabela destino na camada Bronze
map_tables = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

# Itera pelos arquivos da landing e processa apenas os que estão mapeados
for file in files:
    if file in map_tables:
        try:
            dfName = map_tables[file]
            spark.sql(f"DROP TABLE IF EXISTS ecommerce.bronze.{dfName}")
            # Lê o arquivo CSV inferindo a tipagem dos dados
            df = spark.read.csv(path+file, header=True, inferSchema=True)
            # Adiciona uma coluna com o tempo exato da ingestão
            df = df.withColumn("timestamp_ingestion", F.current_timestamp())
            # Salva o dataframe como uma tabela delta, permitindo sobrescrever schema e dados
            df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"ecommerce.bronze.{dfName}")
            print(f"Table {dfName} created")

        except Exception as e:
            print(f"Error creating table {file}: {str(e)}")
    else:
        print(f"O arquivo {file} não faz parte do mapeamento.")

Table tb_customers created
Table tb_geolocalizacao created
Table tb_order_items created
Table tb_order_payments created
Table tb_order_reviews created
Table tb_orders created
Table tb_products created
Table tb_sellers created
Table tb_product_category_name_translation created


In [0]:
# Cria widgets interativos no topo do notebook para receber os parâmetros de data
dbutils.widgets.text(name="data_fim_formatada", defaultValue="09-20-2018", label="MM-DD-AAAA")
dbutils.widgets.text(name="data_inicio_formatada", defaultValue="09-02-2016", label="MM-DD-AAAA")

# Captura os valores informados nos widgets
data_fim_formatada = dbutils.widgets.get("data_fim_formatada")
data_inicio_formatada = dbutils.widgets.get("data_inicio_formatada")

# Faz a requisição a API do Banco Central para pegar a cotação do dólar no período filtrado
apiUrl = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(apiUrl)

if(response.status_code == 200):
    data = response.json()
    # Converte o json diretamente em um dataframe
    df = spark.createDataFrame(data["value"])
    # Adiciona a data de ingestão e salva o resultado na camada bronze
    df = df.withColumn("timestamp_ingestion", F.current_timestamp())
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("ecommerce.bronze.tb_cotacao_dolar")
    print(f"Table created\n")
    spark.table("ecommerce.bronze.tb_cotacao_dolar").display()
else:
    print("API Error!")

Table created



cotacaoCompra,dataHoraCotacao,timestamp_ingestion
3.2425,2016-09-02 13:05:51.688,2026-04-07T20:29:39.866Z
3.2715,2016-09-05 13:09:55.659,2026-04-07T20:29:39.866Z
3.2446,2016-09-06 13:02:39.984,2026-04-07T20:29:39.866Z
3.1928,2016-09-08 13:03:53.968,2026-04-07T20:29:39.866Z
3.2632,2016-09-09 13:14:00.885,2026-04-07T20:29:39.866Z
3.2848,2016-09-12 13:08:01.541,2026-04-07T20:29:39.866Z
3.2966,2016-09-13 13:03:56.534,2026-04-07T20:29:39.866Z
3.3256,2016-09-14 13:05:51.819,2026-04-07T20:29:39.866Z
3.332,2016-09-15 13:08:34.825,2026-04-07T20:29:39.866Z
3.2998,2016-09-16 13:04:11.365,2026-04-07T20:29:39.866Z
